[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/05_humanize_sapiens/05_sapiens_lab.ipynb)


# 05 — BioPhi/Sapiens — 후보 생성 · CDR 사고 · humanness

> 본문 [`05_humanize_sapiens.md`](05_humanize_sapiens.md) 와 **한 절씩 짝지어** 보세요.
> **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 진행합니다.
> 이 노트북의 표·그래프·수치는 **여러분이 방금 돌린 `my_run/`** 에서 나옵니다. 실행을 건너뛰거나 실패하면 저장소에 커밋된 `data/`(실제 실행 산출물) 로 자동 폴백합니다.
>
> **실측 소요 —** Sapiens `predict_scores` VH **0.93초** / VL **0.53초** · humanized 재스코어링 **0.01초**

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`05_humanize_sapiens`) 폴더로 이동한 뒤, 필요한 패키지만 깝니다.
- **로컬**: 이 노트북을 `05_humanize_sapiens/` 폴더 안에서 열었다면 클론 없이 그대로 진행됩니다.

이 셀이 끝나면 두 개의 경로가 준비됩니다 — **`my_run/`**(내가 직접 만들 결과)과 **`data/`**(저장소에 커밋된 레퍼런스 결과).
아래 랩은 항상 `my_run/` 을 먼저 찾고, 없으면 `data/` 로 폴백하면서 **어느 쪽을 쓰는지 print** 합니다.
- ANARCI 는 numbering 을 **hmmscan(HMMER)** 서브프로세스로 돌립니다. Colab 에서는 `apt-get install -y hmmer` 가 함께 실행됩니다 — pip 만으로는 `hmmscan` 이 없어 죽습니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "05_humanize_sapiens"
APT_PKGS = "hmmer"     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib sapiens anarci abnumber"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막습니다.
# (멈춤은 예외가 안 나서 폴백이 안 걸립니다 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어집니다)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

# ANARCI 는 numbering 을 hmmscan(HMMER) 서브프로세스로 돌립니다 — pip 만으로는 hmmscan 이 없습니다.
if APT_PKGS and IN_COLAB:
    _run("apt-get -qq update")                 # 인덱스가 낡으면 install 이 404 로 죽는다
    _run(f"apt-get -qq install -y {APT_PKGS}")
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — Sapiens humanization (본문 5.2)

Sapiens 의 핵심은 `predict_scores` 입니다 — position 마다 20개 아미노산에 대한 **사람 모델의 확률 분포**를 줍니다.
각 자리에서 확률이 가장 높은 잔기를 고르면(argmax) 그게 곧 Sapiens-humanized 서열입니다.

**가드 없이** 돌립니다 — 본문이 경고한 사고를 직접 재현하기 위해서입니다.

In [ ]:
import pandas as pd, numpy as np

def mutations(par, hum):
    return [{"position_1based": i + 1, "wt": a, "mut": b, "mutation": f"{a}{i+1}{b}"}
            for i, (a, b) in enumerate(zip(par, hum)) if a != b]

HAVE_SAPIENS = _have("sapiens")
# 첫 실행은 HuggingFace 에서 Sapiens 가중치(~2MB)를 받습니다. HF 가 느리면 실패할 수 있어
# 두 번까지 재시도하고, 그래도 안 되면 레퍼런스(data/)로 넘어갑니다 — 실습은 끊기지 않습니다.
for _attempt in range(1, 3):
    if not HAVE_SAPIENS:
        break
    try:
        import sapiens
        t0 = time.time()
        mat_h = sapiens.predict_scores(VH, "H")     # rows=position, cols=20 AA
        mat_l = sapiens.predict_scores(VL, "L")
        print(f"Sapiens predict_scores VH+VL: {time.time()-t0:.2f}초")

        hum_h = "".join(mat_h.columns[mat_h.values.argmax(axis=1)])   # 가드 없는 argmax
        hum_l = "".join(mat_l.columns[mat_l.values.argmax(axis=1)])

        mat_h.to_csv(MY / "score_matrix_VH_parental.csv", index_label="position0based")
        mat_l.to_csv(MY / "score_matrix_VL_parental.csv", index_label="position0based")
        write_fasta(MY / "sapiens_humanized_noguard.fasta",
                    {"sapiens_humanized_VH": hum_h, "sapiens_humanized_VL": hum_l})
        mut_h = pd.DataFrame(mutations(VH, hum_h)); mut_h.to_csv(MY / "mutations_VH.csv", index=False)
        mut_l = pd.DataFrame(mutations(VL, hum_l)); mut_l.to_csv(MY / "mutations_VL.csv", index=False)
        print("→", MY / "sapiens_humanized_noguard.fasta")
        break
    except Exception as e:
        print(f"Sapiens 실행 실패({_attempt}/2):", type(e).__name__, str(e)[:160])
        if _attempt == 1:
            print("  · HuggingFace 다운로드 지연일 수 있어 5초 뒤 다시 시도합니다")
            time.sleep(5)
        else:
            HAVE_SAPIENS = False

if not HAVE_SAPIENS:
    print("[레퍼런스] data/ 의 Sapiens 실행 산출물로 진행합니다")
    f = read_fasta("data/sapiens_humanized.fasta")
    hum_h, hum_l = f["sapiens_humanized_VH"], f["sapiens_humanized_VL"]
    mat_h = pd.read_csv("data/score_matrix_VH_parental.csv", index_col=0)
    mat_l = pd.read_csv("data/score_matrix_VL_parental.csv", index_col=0)
    mut_h = pd.read_csv("data/mutations_VH.csv"); mut_l = pd.read_csv("data/mutations_VL.csv")

print(f"\nVH mutation {len(mut_h)}개 · VL mutation {len(mut_l)}개")
print("\nPARENTAL :", VH)
print("SAPIENS  :", hum_h)
print("muts     :", ", ".join(mut_h["mutation"]))

## 2) 내 결과 확인 — **CDR 사고 재현** (본문 5.3)

Ch.04 에서 못 박은 CDR 보호 좌표(`cdr_guard.json`)를 가져와, 방금 나온 mutation 중 **CDR 안에 떨어진 것**을 찾습니다.
Ch.04 를 건너뛰었다면 이 챕터 `data/cdr_table_imgt.csv` 로 폴백합니다.

In [ ]:
# CDR 좌표 — Ch.04 에서 내가 만든 것 우선
gp = GUIDE_ROOT / "04_sequence_qc" / "my_run" / "cdr_guard.json"
if gp.exists():
    print(f"[내 결과 · 04_sequence_qc] {gp}")
    guard = json.loads(gp.read_text())
else:
    print("[레퍼런스] data/cdr_table_imgt.csv 에서 CDR 좌표를 복원합니다")
    ct = pd.read_csv("data/cdr_table_imgt.csv")
    guard = {"H": [], "L": []}
    for _, r in ct.iterrows():
        seq = VH if r["chain"] == "H" else VL
        st = seq.find(r["sequence"]) + 1
        guard[r["chain"]] += list(range(st, st + len(r["sequence"])))

cdr_h = mut_h[mut_h.position_1based.isin(guard["H"])]
cdr_l = mut_l[mut_l.position_1based.isin(guard["L"])]
print(f"\nCDR 안에 떨어진 mutation — VH {len(cdr_h)}개 · VL {len(cdr_l)}개")
display(cdr_l)
print("VL CDR1 =", "".join(VL[p-1] for p in sorted(guard["L"])[:9]),
      "→ 가드 없는 argmax 가 CDR-L1 을 통째로 갈아엎었습니다.")
print("이건 결합력을 깰 수 있는 위험한 mutation 입니다 — 그래서 CDR 좌표를 미리 못 박아야 합니다.")

## 3) 직접 실행 — CDR 가드 적용본 만들기 (본문 5.3 주의)

실무 처방은 셋 중 하나입니다 — ① CDR 을 mask 에서 제외, ② 도구의 CDR 보호 모드, ③ **후처리로 CDR 을 parental 로 되돌리기**.
여기서는 ③ 을 그대로 구현합니다(가장 단순하고 검증하기 쉬움).

In [ ]:
def cdr_guarded(par, hum, protected):
    s = list(hum)
    for p in protected:
        s[p - 1] = par[p - 1]                 # CDR 잔기는 parental 로 복원
    return "".join(s)

g_h = cdr_guarded(VH, hum_h, guard["H"])
g_l = cdr_guarded(VL, hum_l, guard["L"])
write_fasta(MY / "sapiens_humanized_cdrguard.fasta",
            {"sapiens_guarded_VH": g_h, "sapiens_guarded_VL": g_l})

cmp_rows = []
for tag, par, ng, gd, pr in (("VH", VH, hum_h, g_h, guard["H"]), ("VL", VL, hum_l, g_l, guard["L"])):
    cmp_rows.append({
        "체인": tag,
        "가드 없음 · 총 mutation": sum(a != b for a, b in zip(par, ng)),
        "가드 없음 · CDR mutation": sum(par[p-1] != ng[p-1] for p in pr),
        "가드 적용 · 총 mutation": sum(a != b for a, b in zip(par, gd)),
        "가드 적용 · CDR mutation": sum(par[p-1] != gd[p-1] for p in pr),
    })
display(pd.DataFrame(cmp_rows))
print("→", MY / "sapiens_humanized_cdrguard.fasta", "(Ch.08 구조 검증에서 이 파일을 쓸 수 있습니다)")

## 4) 직접 실행 — humanness, **정의를 못 박고** 계산 (본문 5.4)

여기가 본문에서 가장 헷갈리는 지점입니다. "humanized humanness" 는 두 가지로 계산될 수 있고, **값이 다릅니다.**

| 정의 | 계산 | 뜻 |
|---|---|---|
| **(a) argmax-on-parental** | parental 문맥의 확률행렬에서 position 별 **최대 확률**의 평균 | "모델이 각 자리에서 가장 자신 있는 값" — humanized 서열을 **다시 스코어링하지는 않음** |
| **(b) 재스코어링 self-prob** | humanized 서열을 `predict_scores` 에 **다시 넣어**, 그 서열 **자기 잔기**의 확률 평균 | "만들어진 서열이 얼마나 사람다운가" |

**본문 표의 0.815 / 0.872 는 (b) 입니다.** (a) 로 계산하면 0.782 / 0.851 이 나옵니다 — 둘 다 직접 계산해 확인합니다.

In [ ]:
if HAVE_SAPIENS:
    def mean_self_prob(seq, chain):
        m = sapiens.predict_scores(seq, chain)
        return float(np.mean([m.loc[i, aa] for i, aa in enumerate(seq)]))

    rows = []
    for tag, par, ng, gd, chain, mat in (("VH", VH, hum_h, g_h, "H", mat_h),
                                         ("VL", VL, hum_l, g_l, "L", mat_l)):
        par_p = mean_self_prob(par, chain)                       # parental self-prob
        def_a = float(np.mean(mat.values.max(axis=1)))           # (a) parental 행렬의 per-position max
        def_b = mean_self_prob(ng, chain)                        # (b) humanized 재스코어링 ← 본문의 값
        grd_b = mean_self_prob(gd, chain)                        # CDR 가드 적용본 (b)
        rows.append({"chain": tag, "parental": round(par_p, 4),
                     "(a) argmax-on-parental": round(def_a, 4),
                     "(b) 재스코어링 humanized": round(def_b, 4),
                     "(b) CDR 가드 적용본": round(grd_b, 4)})
    hn = pd.DataFrame(rows)
    hn.to_csv(MY / "humanness_summary.csv", index=False)
else:
    print("[레퍼런스] data/humanness_summary.csv (Sapiens 실측)")
    ref = pd.read_csv("data/humanness_summary.csv")
    piv = ref.pivot(index="chain", columns="definition", values="mean_probability")
    hn = pd.DataFrame({
        "chain": piv.index,
        "parental": piv["parental"].round(4).values,
        "(a) argmax-on-parental": piv["definition_a_argmax_on_parental_matrix"].round(4).values,
        "(b) 재스코어링 humanized": piv["definition_b_rescored_humanized"].round(4).values,
        "(b) CDR 가드 적용본": [float("nan")] * len(piv),      # 가드 적용본은 직접 돌려야 나옵니다
    })

display(hn)
print("본문 표(0.694→0.815 / 0.770→0.872)는 parental → **(b)** 열입니다.")
print("(a) 로 계산하면 0.782 / 0.851 — 같은 실행, 다른 정의, 다른 값입니다.")
print("CDR 가드를 적용하면 humanness 는 조금 내려갑니다 — CDR 을 사람화하지 않았으니 당연하고, 그게 맞습니다.")

## 5) 그래프 — parental vs humanized (공용 모듈)

`humanization_viz.humanness_bars` 로 체인별 비교를 그립니다.

In [ ]:
from humanization_viz import humanness_bars

bars = [{"chain": r["chain"], "parental": r["parental"], "humanized": r["(b) 재스코어링 humanized"]}
        for _, r in hn.iterrows()]
humanness_bars(bars, "Sapiens humanness — parental vs humanized (정의 b)", "05_humanness.png")
from IPython.display import Image; Image("05_humanness.png")

## 6) 레퍼런스 대조

`data/` 는 실제 실행 산출물입니다. mutation 리스트가 **문자 단위로 같은지**, humanness 가 같은지 봅니다.

In [ ]:
ref_mut_h = pd.read_csv("data/mutations_VH.csv")
ref_hn    = pd.read_csv("data/humanness_summary.csv")

same = list(ref_mut_h["mutation"]) == list(mut_h["mutation"])
print("VH mutation 리스트 일치:", same, f"(내 결과 {len(mut_h)}개 / 레퍼런스 {len(ref_mut_h)}개)")
if not same:
    print("  내 결과 :", ", ".join(mut_h["mutation"]))
    print("  레퍼런스:", ", ".join(ref_mut_h["mutation"]))

print("\n[레퍼런스 humanness — 정의별]")
display(ref_hn[["chain", "definition", "mean_probability"]])
print("→ definition_b_rescored_humanized 가 본문의 0.8152 / 0.8718 입니다.")

## 이 랩에서 확인한 것

1. Sapiens humanization = `predict_scores` 의 **position 별 argmax**. 실측 **VH 21 · VL 17 mutation**.
2. **가드 없이 돌리면 CDR-L1 이 부서집니다**(`H31A K32Y F33N P34D`) — Ch.04 의 보호 좌표로 후처리 복원하면 CDR mutation 0개.
3. humanness 는 **정의를 밝혀야** 합니다. 본문의 **0.694 → 0.815 (VH)** · **0.770 → 0.872 (VL)** 는 **(b) humanized 재스코어링 self-prob**. 같은 실행에서 (a) 로 계산하면 0.782 / 0.851 로 **다릅니다**.
4. 가드 적용본은 humanness 가 조금 낮습니다 — CDR 을 그대로 뒀으니 당연하며, 결합력을 지키는 대가입니다.

다음 → **[Ch.06 — Humatch · AnthroAb](../06_cdr_safe_tools/06_tools_lab.ipynb)**